# CatVision AI — Model Training Notebook

Welcome to the training notebook for **CatVision AI**! This notebook will download the **Gano Cat Breed Image Collection** (15 breeds, ~5600 images), preprocess the images, and train a ResNet-50 model using PyTorch.

### ⚠️ IMPORTANT
Before running any cells, make sure you are using a GPU runtime:
1. In the top menu, go to **Runtime** → **Change runtime type**.
2. Select **T4 GPU** (or any available GPU) under Hardware Accelerator.
3. Click **Save**.

## 1. Install & Import Dependencies

In [ ]:
# Install standard packages
!pip install torch torchvision matplotlib tqdm kaggle

In [ ]:
import os
import time
import copy
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import datasets, models, transforms

## 2. Mount Google Drive (Optional)
Uncomment and run this cell if you want to mount your Google Drive to save checkpoints directly to your drive storage.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## 3. Download & Extract Gano Dataset
We use the Gano Cat Breed Image Collection from Kaggle. Note: You will need your `kaggle.json` uploaded to Colab for the CLI to authenticate.

In [ ]:
# Set Kaggle config dir to where we upload kaggle.json if necessary
# import os
# os.environ['KAGGLE_CONFIG_DIR'] = '/content'

!kaggle datasets download -d shawngano/gano-cat-breed-image-collection -p ./data

zip_path = "./data/gano-cat-breed-image-collection.zip"
extract_path = "./data/gano"

if os.path.exists(zip_path):
    print("Unzipping dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Unzipping complete.")
else:
    print("Warning: Zip file not found. Ensure the kaggle command worked.")

## 4. Dataset Loading & Transformations
We apply data augmentation for training, and setup standard normalization for validation.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Find the actual folder containing the breeds
data_dir = "./data/gano"
for root, dirs, files in os.walk(extract_path):
    if len(dirs) >= 10:
        data_dir = root
        break

# Create two datasets with different transforms
dataset_train = datasets.ImageFolder(root=data_dir, transform=train_transform)
dataset_val = datasets.ImageFolder(root=data_dir, transform=val_transform)

print(f"Found {len(dataset_train.classes)} breeds: {dataset_train.classes}")
print(f"Total images: {len(dataset_train)}")

# Split 85% train, 15% val
dataset_size = len(dataset_train)
indices = list(range(dataset_size))
np.random.shuffle(indices)
split = int(np.floor(0.15 * dataset_size))

train_indices, val_indices = indices[split:], indices[:split]

train_dataset = Subset(dataset_train, train_indices)
val_dataset = Subset(dataset_val, val_indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

## 5. Define Model (ResNet-50 Transfer Learning)
We load a pre-trained ResNet-50, freeze early layers, and replace the classification head to output our classes.

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Freeze early layers
for param in model.parameters():
    param.requires_grad = False

# Replace classification layer
num_features = model.fc.in_features
num_classes = len(dataset_train.classes)

model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, num_classes)
)

# Unfreeze layer3 and layer4 for fine-tuning
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model configured on: {device}")

## 6. Training Pipeline

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=7, gamma=0.1)

EPOCHS = 35
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    # Training mode
    model.train()
    running_loss = 0.0
    corrects = 0
    total = 0
    
    for inputs, labels in tqdm(train_loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        corrects += torch.sum(preds == labels.data)
        total += inputs.size(0)
        
    scheduler.step()  # Step the scheduler at the end of the train phase
    epoch_loss = running_loss / total
    epoch_acc = corrects.double() / total * 100
    print(f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.2f}%")
    
    # Validation mode
    model.eval()
    val_loss = 0.0
    val_corrects = 0
    val_total = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validation"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_corrects += torch.sum(preds == labels.data)
            val_total += inputs.size(0)
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_corrects.double() / val_total * 100
    print(f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.2f}%")
    
    # Save best model weight weights
    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

print(f"\nTraining complete! Best Validation Accuracy: {best_acc:.2f}%")
model.load_state_dict(best_model_wts)

## 7. Save & Download Model
This saves the state dict locally and triggers a browser download.

In [ ]:
model_filename = "ganomodel.pth"
torch.save(model.state_dict(), model_filename)
print(f"Saved state dict to {model_filename}")

# Trigger download in browser
from google.colab import files
files.download(model_filename)